In [0]:
WITH gaming_txn AS (
  SELECT
    VisitDateId,
    Net_Revenue,
    Quantity
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction as t
  left join cpx_dataanalytics_gold.customer360.cpx_d_item as I
  on I.itemskey = t.ItemSkey
  WHERE LocationId = "7021"
    AND I.Item_Class_Description1 IN (
      'TRR - AMUSEMENT - GAMES',
      'TRR - AMUSEMENT - PHYSICAL GAMES',
      'TRR - PROMO - AMUSEMENT'
    )
    AND (
      VisitDateId BETWEEN 20250601 AND 20250831
      OR VisitDateId BETWEEN 20260601 AND 20260831
    )
),
fnb_txn AS (
  SELECT
    VisitDateId,
    Net_Revenue,
    Quantity
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction as t
  left join cpx_dataanalytics_gold.customer360.cpx_d_item as I
  on I.itemskey = t.ItemSkey
  WHERE LocationId = "7021"
    AND I.Item_Class_Description1 IN (
      'PDM - MENU FOOD - CANDY',
      'TRR - INV ALC BEV - BOTTLE',
      'TRR - INV ALC BEV - CAN',
      'TRR - INV ALC BEV - WINE',
      'TRR - MENU ALC BEV - COCKTAIL',
      'TRR - MENU ALC BEV - DRAUGHT BEER',
      'TRR - MENU BEV - NA COLD BEV',
      'TRR - MENU BEV - NA HOT BEV',
      'TRR - MENU FOOD - APP',
      'TRR - MENU FOOD - DESSERT',
      'TRR - MENU FOOD - KIDS',
      'TRR - MENU FOOD - PIZZA',
      'TRR - MENU FOOD - POUTINE',
      'TRR - MENU FOOD - SANDWICHES',
      'TRR - MOD MENU - ADD-ON',
      'TRR - INV BEV - NA BEV',
      'TRR - MENU ALC BEV - LIQUOR',
      'TRR - MENU ALC BEV - WINE',
      'TRR - MENU FOOD - ENTRÉE',
      'TRR - MENU FOOD - SALAD',
      'VHO-LICENSED FOOD',
      'TRR - INV ALC BEV - LIQUOR',
      'VHO-CANDY',
      'VIP Food - Addons',
      'VHO-LICENSED DRINKS',
      'TRR - MENU FOOD - BRUNCH',
      'VHO-DRINK'
    )
    AND (
      VisitDateId BETWEEN 20250601 AND 20250831
      OR VisitDateId BETWEEN 20260601 AND 20260831
    )
),
combined AS (
  SELECT 'Gaming' AS Category, VisitDateId, Net_Revenue, Quantity FROM gaming_txn
  UNION ALL
  SELECT 'FnB' AS Category, VisitDateId, Net_Revenue, Quantity FROM fnb_txn
)
SELECT
  CASE WHEN VisitDateId BETWEEN 20250601 AND 20250630 THEN 'Jun_2025'
       WHEN VisitDateId BETWEEN 20250701 AND 20250731 THEN 'Jul_2025'
       WHEN VisitDateId BETWEEN 20250801 AND 20250831 THEN 'Aug_2025'
       WHEN VisitDateId BETWEEN 20260601 AND 20260630 THEN 'Jun_2026'
       WHEN VisitDateId BETWEEN 20260701 AND 20260731 THEN 'Jul_2026'
       WHEN VisitDateId BETWEEN 20260801 AND 20260831 THEN 'Aug_2026'
  END AS Period,
  Category,
  SUM(Net_Revenue) AS Total_Revenue,
  SUM(Quantity) AS Total_Quantity
FROM combined
WHERE DAYOFWEEK(TO_DATE(CAST(VisitDateId AS STRING), 'yyyyMMdd')) IN (3, 4, 5)
GROUP BY 1, 2
ORDER BY 1, 2